# Agent Harness：Model → Tool → Observation → Model

确定性假模型演示真实 Write/Read 工具，没有网络或旧 session/harness API。


In [ ]:
import tempfile
from pathlib import Path
from fox_ai.src import Model, AssistantMessage, ToolCall, Done
from fox_agent_core.src import Agent, Hooks, coding_tools

async def fake_model(model, context, options):
    last = context.messages[-1]
    if last.role == "user":
        yield Done(AssistantMessage(tool_calls=[ToolCall("w", "Write", {"file_path": "demo.py", "content": "print(42)"})]))
    elif last.name == "Write":
        yield Done(AssistantMessage(tool_calls=[ToolCall("r", "Read", {"file_path": "demo.py"})]))
    else:
        assert "print(42)" in last.content
        yield Done(AssistantMessage("Verified demo.py"))

async def demo():
    with tempfile.TemporaryDirectory() as directory:
        observed = []
        agent = Agent(Model("mock"), coding_tools(directory), stream_fn=fake_model,
                      hooks=Hooks(after_tool=lambda call, result: observed.append(call.name)))
        events = [event async for event in agent.run("Create and read demo.py")]
        assert observed == ["Write", "Read"]
        assert events[-1].data["status"] == "completed"
        print([e.type for e in events])

await demo()


主循环只消费模型、执行工具、追加 result；Hooks 是 callback。对照 src/agent_loop.py 阅读，不需要 session tree、plugin lifecycle、context compaction 的知识。
